# 12 — Generate Falcon 9 paper figures

This is the canonical figure-production notebook for the Falcon 9 Fireball paper.

It reads finalized geometry, waveform, measurement, and array-analysis products
created by earlier notebooks and generates publication figures without silently
recomputing scientific measurements.

Current figure groups:

1. SLC-40 and BCHH deployment geometry;
2. initial second-stage failure waveforms;
3. principal explosion waveforms;
4. capsule-related acoustic pulses;
5. catalogue array-result summary.

Additional final and supplementary figures can be added here as their upstream
products are frozen.

**Required upstream products**

- standardized geometry products from Notebook 02;
- corrected BCHH analysis waveform;
- key-event pressure measurements;
- event-catalogue array results.


## 1. Imports, project paths, and plotting configuration


In [1]:

from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyproj import CRS, Geod, Transformer

from obspy import Stream, UTCDateTime, read

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

MODULE_DIR = PROJECT_ROOT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from project_config import ensure_output_dirs
from geometry_products import read_geometry_products
from figure1_utils import (
    get_sensor_location,
    make_figure1,
    save_figure,
)
from plotting import plot_key_event_waveforms

PATHS = ensure_output_dirs(PROJECT_ROOT)
DERIVED_DIR = PATHS["derived"]
FIGURE_DIR = PATHS["figures"]
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

WGS84 = CRS.from_epsg(4326)
UTM17N = CRS.from_epsg(32617)

geod = Geod(ellps="WGS84")
ll_to_utm = Transformer.from_crs(WGS84, UTM17N, always_xy=True)
utm_to_ll = Transformer.from_crs(UTM17N, WGS84, always_xy=True)

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})

print(f"Project root: {PROJECT_ROOT}")
print(f"Derived data: {DERIVED_DIR}")
print(f"Figure output: {FIGURE_DIR}")


Project root: /Users/glennthompson/Developer/KSCRocketSeismoHydrology/08_fireball_paper
Derived data: /Users/glennthompson/Developer/KSCRocketSeismoHydrology/08_fireball_paper/outputs/derived
Figure output: /Users/glennthompson/Developer/KSCRocketSeismoHydrology/08_fireball_paper/outputs/figures


## 2. Figure 1 — SLC-40 and BCHH deployment geometry

This section consumes the standardized geometry products written by Notebook 02.
Coordinates are not re-entered manually.


In [2]:
(
    inventory_event,
    channels_df,
    stations_df,
    locations_df,
) = read_geometry_products(DERIVED_DIR)

print("Channel columns:")
print(channels_df.columns.tolist())

print("\nLocation columns:")
print(locations_df.columns.tolist())

display(stations_df)
display(channels_df)
display(locations_df)


FileNotFoundError: [Errno 2] No such file or directory: '/Users/glennthompson/Developer/KSCRocketSeismoHydrology/08_fireball_paper/outputs/derived/event_inventory.xml'

### 2.1 Adapt the normalized channel table for mapping

The three seismic components represent one physical seismometer location and are
collapsed to a single `Seismometer` row.


In [ ]:
CHANNEL_TO_SENSOR = {
    "DHZ": "Seismometer",
    "DHN": "Seismometer",
    "DHE": "Seismometer",
    "HHZ": "Seismometer",
    "HHN": "Seismometer",
    "HHE": "Seismometer",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
    "HD1": "HD1",
    "HD2": "HD2",
    "HD3": "HD3",
}

CHANNEL_TO_LABEL = {
    "DHZ": "BCHH",
    "DHN": "BCHH",
    "DHE": "BCHH",
    "HHZ": "BCHH",
    "HHN": "BCHH",
    "HHE": "BCHH",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
    "HD1": "HD1",
    "HD2": "HD2",
    "HD3": "HD3",
}

required_channel_columns = {
    "channel",
    "latitude",
    "longitude",
}
missing = required_channel_columns.difference(channels_df.columns)
if missing:
    raise KeyError(
        "The standardized channel table is missing required columns: "
        f"{sorted(missing)}"
    )

channels_for_map = channels_df.copy()
channel_codes = (
    channels_for_map["channel"]
    .astype(str)
    .str.upper()
)

channels_for_map["sensor"] = channel_codes.map(CHANNEL_TO_SENSOR)
channels_for_map["label"] = channel_codes.map(CHANNEL_TO_LABEL)
channels_for_map["lat"] = channels_for_map["latitude"].astype(float)
channels_for_map["lon"] = channels_for_map["longitude"].astype(float)

eastings, northings = ll_to_utm.transform(
    channels_for_map["lon"].to_numpy(),
    channels_for_map["lat"].to_numpy(),
)
channels_for_map["easting"] = eastings
channels_for_map["northing"] = northings

unmapped = channels_for_map.loc[
    channels_for_map["sensor"].isna(),
    ["seed_id", "channel", "sensor_description"],
]
if not unmapped.empty:
    print("Ignoring channels that are not used in Figure 1:")
    display(unmapped)

mapped = channels_for_map.dropna(subset=["sensor"]).copy()

seismometer_rows = mapped.loc[
    mapped["sensor"] == "Seismometer"
]
if seismometer_rows.empty:
    raise ValueError("No seismic component was mapped to 'Seismometer'.")

# All three components are co-located; retain one representative row.
seismometer_row = seismometer_rows.iloc[[0]].copy()

infrasound_rows = (
    mapped.loc[mapped["sensor"].isin(["HD1", "HD2", "HD3"])]
    .sort_values("sensor")
    .drop_duplicates(subset=["sensor"])
)

bchh_sensors_df = pd.concat(
    [seismometer_row, infrasound_rows],
    ignore_index=True,
)

expected_sensors = {"Seismometer", "HD1", "HD2", "HD3"}
actual_sensors = set(bchh_sensors_df["sensor"])
if actual_sensors != expected_sensors:
    raise ValueError(
        "Expected exactly these physical sensors: "
        f"{sorted(expected_sensors)}; found {sorted(actual_sensors)}"
    )

figure1_columns = [
    "sensor",
    "label",
    "lat",
    "lon",
    "easting",
    "northing",
    "channel",
    "seed_id",
    "sensor_description",
]
display(bchh_sensors_df[figure1_columns])


### 2.2 Extract launch-pad and BCHH reference locations


In [ ]:
def get_unique_kml_location(
    dataframe: pd.DataFrame,
    kml_id: str,
) -> dict:
    matches = dataframe.loc[
        dataframe["kml_id"]
        .astype(str)
        .str.upper()
        .eq(kml_id.upper())
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one KML placemark {kml_id!r}; "
            f"found {len(matches)}."
        )

    row = matches.iloc[0].copy()

    # Normalize possible coordinate-column conventions.
    lat_key = (
        "lat"
        if "lat" in row.index
        else "latitude"
    )
    lon_key = (
        "lon"
        if "lon" in row.index
        else "longitude"
    )

    return {
        **row.to_dict(),
        "name": str(row.get("name", kml_id)),
        "lat": float(row[lat_key]),
        "lon": float(row[lon_key]),
    }


SLC40 = get_unique_kml_location(locations_df, "SLC40")
SLC41 = get_unique_kml_location(locations_df, "SLC41")

BCHH = get_sensor_location(
    bchh_sensors_df,
    sensor_name="Seismometer",
    display_name="BCHH",
)

print("SLC-40:", SLC40)
print("SLC-41:", SLC41)
print("BCHH:", BCHH)


### 2.3 Generate and save Figure 1


In [ ]:
fig, axes = make_figure1(
    slc40=SLC40,
    slc41=SLC41,
    bchh=BCHH,
    bchh_sensors=bchh_sensors_df,
    geod=geod,
    ll_to_utm=ll_to_utm,
    utm_to_ll=utm_to_ll,
)

output_paths = save_figure(
    fig=fig,
    output_directory=FIGURE_DIR,
    filename_stem="fig01_slc40_bchh_location",
    extensions=("png", "pdf"),
    dpi=300,
)

print("Saved Figure 1:")
for path in output_paths:
    print(f"  {Path(path).resolve()}")

plt.show()


## 3. Load finalized waveform and key-event measurement products

The figure notebook reads measurements from CSV and does not recompute them.


In [ ]:

CORRECTED_WAVEFORM_FILE = DERIVED_DIR / "bchh_corrected_analysis_window.pkl"
KEY_EVENT_MEASUREMENTS_FILE = DERIVED_DIR / "key_event_pressure_measurements.csv"

for required_path, label in [
    (CORRECTED_WAVEFORM_FILE, "corrected waveform"),
    (KEY_EVENT_MEASUREMENTS_FILE, "key-event measurements"),
]:
    if not required_path.exists():
        raise FileNotFoundError(f"{label} not found: {required_path}")

st_corr = read(str(CORRECTED_WAVEFORM_FILE), format="PICKLE")
measurements = pd.read_csv(KEY_EVENT_MEASUREMENTS_FILE)

print(st_corr)
display(measurements)


## 4. Initial second-stage failure


In [ ]:

EXPLOSION_TIME = UTCDateTime("2016-09-01T13:07:12.080")
t0, t1 = EXPLOSION_TIME + 3.0 - 0.08, EXPLOSION_TIME + 5.0 - 0.08
st_event = st_corr.copy().trim(t0, t1)
event_results = measurements.loc[
    measurements["event"] == "Initial second-stage failure"
]
plot_key_event_waveforms(
    st_event,
    reference_time=t0,
    pressure_results=event_results,
    title="Initial second-stage failure",
    outfile=FIGURE_DIR / "second_stage_waveforms.png",
)
plt.show()


## 5. Principal explosion


In [ ]:

t0, t1 = EXPLOSION_TIME + 6.0 - 0.08, EXPLOSION_TIME + 9.0 - 0.08
st_event = st_corr.copy().trim(t0, t1)
event_results = measurements.loc[
    measurements["event"] == "Principal explosion"
]
plot_key_event_waveforms(
    st_event,
    reference_time=t0,
    pressure_results=event_results,
    title="Principal Falcon 9 explosion",
    outfile=FIGURE_DIR / "principal_explosion_waveforms.png",
)
plt.show()


## 6. Capsule-related acoustic pulses


In [ ]:

t0 = UTCDateTime("2016-09-01T13:07:27.0")
t1 = UTCDateTime("2016-09-01T13:07:30.0")
st_event = st_corr.copy().trim(t0, t1)
capsule_results = measurements.loc[
    measurements["event"].isin(["Capsule pulse 1", "Capsule pulse 2"])
]
# For a two-pulse figure, marker annotations should be added manually or by
# extending plotting.py to accept multiple measurement windows.
plot_key_event_waveforms(
    st_event,
    reference_time=t0,
    pressure_results=None,
    title="Capsule-related acoustic pulses",
    outfile=FIGURE_DIR / "capsule_waveforms.png",
    add_measurements=False,
)
plt.show()


## 7. Catalogue array-result summary

This section reads the finalized array-results table. If the expected result file
does not exist, the notebook reports which upstream analysis must be run.


In [ ]:

ARRAY_RESULTS_CANDIDATES = [
    DERIVED_DIR / "planar_array_results.csv",
    DERIVED_DIR / "event_catalogue_array_results.csv",
]

array_file = next((path for path in ARRAY_RESULTS_CANDIDATES if path.exists()), None)

if array_file is None:
    print(
        "No finalized array-results table found. Expected one of:\n  "
        + "\n  ".join(str(path) for path in ARRAY_RESULTS_CANDIDATES)
    )
else:
    array_results = pd.read_csv(array_file)
    print(f"Loaded array results: {array_file}")
    display(array_results.head())

    required_columns = {"back_azimuth_deg"}
    missing = required_columns.difference(array_results.columns)
    if missing:
        raise KeyError(
            f"Array-results table is missing required columns: {sorted(missing)}"
        )

    color_column = (
        "mean_abs_correlation"
        if "mean_abs_correlation" in array_results.columns
        else None
    )

    fig, ax = plt.subplots(figsize=(10, 4))

    if color_column is not None:
        scatter = ax.scatter(
            np.arange(len(array_results)),
            array_results["back_azimuth_deg"],
            c=array_results[color_column],
            s=22,
        )
        cbar = fig.colorbar(scatter, ax=ax)
        cbar.set_label("Mean absolute correlation")
    else:
        ax.scatter(
            np.arange(len(array_results)),
            array_results["back_azimuth_deg"],
            s=22,
        )

    ax.set_xlabel("Event number")
    ax.set_ylabel("Back azimuth (degrees)")
    ax.set_title("Source direction through the explosion sequence")
    ax.grid(True, alpha=0.25)

    for ext in ("png", "pdf"):
        outfile = FIGURE_DIR / f"catalogue_back_azimuth.{ext}"
        fig.savefig(outfile, dpi=300, bbox_inches="tight")
        print(f"Saved: {outfile}")

    plt.show()


## 8. Figure manifest

The manifest records the files expected from the currently implemented figure
sections. Additional figures should be added here as the manuscript figure set is
finalized.


In [ ]:

figure_manifest = pd.DataFrame([
    {
        "figure": "Figure 1",
        "description": "SLC-40 and BCHH deployment geometry",
        "png": FIGURE_DIR / "fig01_slc40_bchh_location.png",
        "pdf": FIGURE_DIR / "fig01_slc40_bchh_location.pdf",
    },
    {
        "figure": "Key event",
        "description": "Initial second-stage failure waveforms",
        "png": FIGURE_DIR / "second_stage_waveforms.png",
        "pdf": None,
    },
    {
        "figure": "Key event",
        "description": "Principal explosion waveforms",
        "png": FIGURE_DIR / "principal_explosion_waveforms.png",
        "pdf": None,
    },
    {
        "figure": "Key event",
        "description": "Capsule-related acoustic pulses",
        "png": FIGURE_DIR / "capsule_waveforms.png",
        "pdf": None,
    },
    {
        "figure": "Array result",
        "description": "Catalogue back azimuth",
        "png": FIGURE_DIR / "catalogue_back_azimuth.png",
        "pdf": FIGURE_DIR / "catalogue_back_azimuth.pdf",
    },
])

figure_manifest["png_exists"] = figure_manifest["png"].map(
    lambda value: Path(value).exists() if value is not None else False
)
figure_manifest["pdf_exists"] = figure_manifest["pdf"].map(
    lambda value: Path(value).exists() if value is not None else False
)

display(figure_manifest)

manifest_file = FIGURE_DIR / "figure_manifest.csv"
figure_manifest.astype(str).to_csv(manifest_file, index=False)
print(f"Saved manifest: {manifest_file}")


## Outputs

This notebook currently generates:

- `fig01_slc40_bchh_location.png/.pdf`;
- `second_stage_waveforms.png`;
- `principal_explosion_waveforms.png`;
- `capsule_waveforms.png`;
- `catalogue_back_azimuth.png/.pdf`;
- `figure_manifest.csv`.

The synchronized audio/reduced-time chronology, event-catalogue dashboard, phase
summary, and other final figures should be added here only after their upstream
analysis products and plotting functions are stabilized.
